In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from db_config import DB_URL

engine = create_engine(DB_URL)

In [2]:
df = pd.read_sql("""
    SELECT f.*, d.sex, d.age, d.address, d.medu, d.fedu,
           d.internet, d.famsup, d.health
    FROM fact_grades f
    JOIN dim_student d USING (student_id)
""", engine)

In [ ]:
# 1. GPA DISTRIBUTION — percentiles
gpa_vals = df["gpa"].dropna().values
p = np.percentile(gpa_vals, [10, 25, 50, 75, 90]) #type: ignore
labels = ["P10","P25","P50","P75","P90"]
print("=== GPA distribution ===")
for l, v in zip(labels, p):
    print(f"  {l}: {v:.2f}/20")

=== GPA distribution ===
  P10: 7.33/20
  P25: 9.33/20
  P50: 11.33/20
  P75: 13.33/20
  P90: 15.33/20


In [4]:
# 2. GRADE TREND ANALYSIS — who is improving vs declining?
print("\n=== Grade trend breakdown ===")
df["trend_label"] = np.select(
    [
        df["grade_trend"] > 2,
        df["grade_trend"] < -2,
    ],
    ["improving", "declining"],
    default="stable"
)

trend_counts = df["trend_label"].value_counts()
for label, count in trend_counts.items():
    pct = count / len(df) * 100
    print(f"  {label:<12}: {count:>4}  ({pct:.1f}%)")


=== Grade trend breakdown ===
  stable      :  901  (86.3%)
  improving   :   78  (7.5%)
  declining   :   65  (6.2%)


In [5]:
# 3. CORRELATION — which factors correlate with final grade?
numeric_factors = ["studytime","failures","absences",
                   "medu","fedu","health","age"]

print("\n=== Factor correlation with G3 (final grade) ===")
correlations = {}
for col in numeric_factors:
    vals = df[[col,"g3"]].dropna()
    r = np.corrcoef(vals[col].values, vals["g3"].values)[0,1]  #type: ignore
    correlations[col] = round(r, 3)

for col, r in sorted(correlations.items(),
                     key=lambda x: abs(x[1]),
                     reverse=True):
    direction = "↑" if r > 0 else "↓"
    print(f"  {col:<15}: r = {r:>+.3f}  {direction}")


=== Factor correlation with G3 (final grade) ===
  failures       : r = -0.383  ↓
  medu           : r = +0.201  ↑
  studytime      : r = +0.162  ↑
  fedu           : r = +0.160  ↑
  age            : r = -0.125  ↓
  health         : r = -0.080  ↓
  absences       : r = -0.046  ↓


In [8]:
# 4. Z-SCORE — flag students far below class average
print("\n=== Below-average students by z-score ===")
mean = np.mean(df["gpa"].values)  #type: ignore
std  = np.std(df["gpa"].values)   #type: ignore
df["gpa_zscore"] = (df["gpa"].values - mean) / std

low_performers = df[df["gpa_zscore"] < -1.5].shape[0]
print(f"  Students with GPA z-score < -1.5 : {low_performers}")
print(f"  ({low_performers/len(df)*100:.1f}% of total)")


=== Below-average students by z-score ===
  Students with GPA z-score < -1.5 : 72
  (6.9% of total)


In [9]:
# 5. ABSENCE IMPACT — bin absences and compute pass rate
print("\n=== Absence buckets vs pass rate ===")
bins   = [0, 3, 7, 15, 100]
labels_bin = ["0-3","4-7","8-15","16+"]
df["absence_bucket"] = pd.cut(
    df["absences"], bins=bins, labels=labels_bin, right=True
)

absence_analysis = (
    df.groupby("absence_bucket")
    .agg(
        students   = ("passed", "count"),
        pass_rate  = ("passed", "mean"),
        avg_g3     = ("g3",     "mean")
    )
    .reset_index()
)
absence_analysis["pass_rate"] = np.round(
    absence_analysis["pass_rate"] * 100, 1
)
absence_analysis["avg_g3"] = np.round(
    absence_analysis["avg_g3"], 2
)
print(absence_analysis.to_string(index=False))


=== Absence buckets vs pass rate ===
absence_bucket  students  pass_rate  avg_g3
           0-3       205       83.9   12.04
           4-7       253       82.6   11.84
          8-15       173       71.7   11.05
           16+        54       63.0   10.31
